# Electrothermal Coupling with an External Thermal Model

`CellElectrical` uses PyBaMM's **isothermal** electrochemistry and exposes heat generation
as an output. Wiring it to `LumpedThermal` creates a closed electrothermal feedback loop
directly in PathSim — useful for pack-level thermal networks or custom cooling models.


## Model

`LumpedThermal` implements a single-node energy balance:

$$
m C_p \frac{dT}{dt} = \dot{Q} - UA\,(T - T_{\mathrm{amb}})
$$

`CellElectrical` outputs total heat generation [W], which `LumpedThermal` accepts
directly — no unit bridging needed.


In [ ]:
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Constant, Scope
from pathsim.solvers import ESDIRK43

from pathsim_batt import CellElectrical, LumpedThermal

## Simulation: 1 C Discharge with Electrothermal Feedback


In [ ]:
C_nom       = 5.0    # [Ah]
I_discharge = 1.0 * C_nom
T_amb0      = 298.15 # [K]

mass = 0.065  # [kg]
Cp   = 750.0  # [J kg⁻¹ K⁻¹]
UA   = 0.5    # [W K⁻¹]

cell    = CellElectrical(initial_soc=1.0)
thermal = LumpedThermal(mass=mass, Cp=Cp, UA=UA, T0=T_amb0)
I_src   = Constant(I_discharge)
T_src   = Constant(T_amb0)
sco     = Scope(labels=["V", "SOC", "T_cell"])

sim = Simulation(
    blocks=[I_src, T_src, cell, thermal, sco],
    connections=[
        Connection(I_src,          cell["I"]),
        Connection(thermal["T"],   cell["T_cell"]),
        Connection(cell["Q_dot"],  thermal["Q_dot"]),
        Connection(T_src,          thermal["T_amb"]),
        Connection(cell["V"],      sco[0]),
        Connection(cell["SOC"],    sco[1]),
        Connection(thermal["T"],   sco[2]),
    ],
    dt=10.0,
    Solver=ESDIRK43,
)

sim.run(1800.0)
t, [V, SOC, T_cell] = sco.read()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)

axes[0].plot(t / 3600, V, color="steelblue")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_ylim(3.0, 4.3)

axes[1].plot(t / 3600, T_cell - 273.15, color="orangered")
axes[1].axhline(T_amb0 - 273.15, color="grey", linestyle="--",
                linewidth=0.8, label="$T_{\\mathrm{amb}}$")
axes[1].set_ylabel("Cell temperature / °C")
axes[1].legend()

axes[2].plot(t / 3600, SOC * 100, color="forestgreen")
axes[2].set_ylabel("SOC / %")
axes[2].set_ylim(0, 105)
axes[2].set_xlabel("Time / h")

fig.suptitle("1 C Discharge — External Electrothermal Coupling (Chen2020)", fontsize=12)
plt.tight_layout()
plt.show()


## Effect of Cooling Conditions

Sweep of $UA$ from adiabatic to liquid-cooled to show the impact on voltage and temperature.


In [ ]:
ua_scenarios = {
    "Adiabatic (UA = 0)":          0.0,
    "Moderate (UA = 0.5 W K⁻¹)":  0.5,
    "Aggressive (UA = 5 W K⁻¹)":  5.0,
}
colors_ua  = ["firebrick", "steelblue", "teal"]
ua_results = {}

for (label, ua_val), clr in zip(ua_scenarios.items(), colors_ua):
    I_src_i   = Constant(I_discharge)
    T_src_i   = Constant(T_amb0)
    cell_i    = CellElectrical(initial_soc=1.0)
    thermal_i = LumpedThermal(mass=mass, Cp=Cp, UA=ua_val, T0=T_amb0)
    sco_i     = Scope(labels=["V", "T", "SOC"])

    sim_i = Simulation(
        blocks=[I_src_i, T_src_i, cell_i, thermal_i, sco_i],
        connections=[
            Connection(I_src_i,          cell_i["I"]),
            Connection(thermal_i["T"],   cell_i["T_cell"]),
            Connection(cell_i["Q_dot"],  thermal_i["Q_dot"]),
            Connection(T_src_i,          thermal_i["T_amb"]),
            Connection(cell_i["V"],      sco_i[0]),
            Connection(thermal_i["T"],   sco_i[1]),
            Connection(cell_i["SOC"],    sco_i[2]),
        ],
        dt=10.0,
        Solver=ESDIRK43,
    )

    sim_i.run(1800.0)
    t_i, [V_i, T_i, SOC_i] = sco_i.read()
    ua_results[label] = {"t": t_i, "V": V_i, "T": T_i, "SOC": SOC_i, "color": clr}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for label, res in ua_results.items():
    axes[0].plot(res["SOC"] * 100, res["V"],             label=label, color=res["color"])
    axes[1].plot(res["t"] / 3600,  res["T"] - 273.15,   label=label, color=res["color"])

axes[0].set_xlabel("SOC / %")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_xlim(0, 100)
axes[0].invert_xaxis()
axes[0].set_ylim(3.0, 4.3)
axes[0].legend(fontsize=8)
axes[0].set_title("Voltage vs. SOC")

axes[1].set_xlabel("Time / h")
axes[1].set_ylabel("Cell temperature / °C")
axes[1].axhline(T_amb0 - 273.15, color="grey", linestyle=":",
                linewidth=0.8, label="$T_{\\mathrm{amb}}$")
axes[1].legend(fontsize=8)
axes[1].set_title("Temperature Rise vs. Cooling Intensity")

fig.suptitle("Effect of Cooling Condition — 1 C Discharge (Chen2020)", fontsize=12)
plt.tight_layout()
plt.show()


## Summary

- `CellElectrical` + `LumpedThermal` form a closed electrothermal feedback loop; PathSim resolves the coupling at each step.
- `CellElectrical` outputs total heat [W], which connects directly to `LumpedThermal`'s input — no unit conversion needed.
- Stronger cooling keeps the cell cooler and slightly raises the discharge voltage.
- For DAE models (e.g. DFN), see notebook 03.
